# TensorRT INT8 Engine Build + Benchmark (Google Colab · T4)

**GPU ONLY.** Mirrors `src/optimization/build_tensorrt.py` and `src/optimization/benchmark.py`. The CPU demo never uses TensorRT — it uses the ONNX-INT8 model. This notebook is for producing the TensorRT-INT8 row of the README results table.

## Colab run steps
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload (or mount from Drive) your `best.onnx` and a folder of calibration images (e.g. `VisDrone-DET/images/val`).
3. *Runtime → Run all*.
4. Paste the printed size / latency / FPS / mAP-delta into the README table.

In [ ]:
# --- Verify GPU ---
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
!nvcc --version | tail -1

In [ ]:
# --- Install GPU optimization deps ---
# Colab usually ships a compatible TensorRT with the CUDA runtime; if the
# import fails, install a matching wheel from the NVIDIA pip index.
!pip -q install onnx onnxruntime-gpu pycuda
try:
    import tensorrt as trt
    print('TensorRT', trt.__version__)
except ImportError:
    !pip -q install tensorrt
    import tensorrt as trt
    print('TensorRT', trt.__version__)

In [ ]:
# --- Get the repo (replace with your fork URL) ---
# !git clone https://github.com/<you>/object-detection-tracking.git
# %cd object-detection-tracking
#
# Expected inputs (upload or copy from Drive):
#   weights/best.onnx
#   data/yolo/VisDrone-DET/images/val/   (calibration images)
ONNX = 'weights/best.onnx'
CALIB_DIR = 'data/yolo/VisDrone-DET/images/val'
ENGINE = 'weights/best.int8.engine'

In [ ]:
# --- Build the INT8 engine (calibrated) ---
!python -m src.optimization.build_tensorrt \
    --onnx {ONNX} --calib-dir {CALIB_DIR} --engine {ENGINE} --num-samples 200

In [ ]:
# --- Benchmark FP32 vs ONNX-INT8 vs TensorRT-INT8 ---
# (run on the GPU box so the TensorRT row is measured, not skipped)
!python -m src.optimization.benchmark --weights-dir weights --runs 100 --out trt_results.md
print(open('trt_results.md').read())

## After building

Copy the printed table into the README **Optimization** section. Targets to compare against: ~68% size reduction, <1.5% mAP loss from INT8, and the TensorRT speedup over FP32. Report the real measured numbers.